# 📚 Define Libraries

In [1]:
from google.cloud import bigquery
from datetime import datetime

# ---------------------------------------------------------
# 1. Create BigQuery client
# ---------------------------------------------------------
client = bigquery.Client(project="prj-prod-dataplatform")

C:\Users\Dwaipayan\AppData\Roaming\Python\Python312\site-packages\google\auth\_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


# ❄️ alpha_cash_stack_model_v1_2

In [2]:
print("Starting BigQuery job...")

sq = """ 
create or replace table `dap_ds_poweruser_playground.alpha_cash_stack_model_v1_2` as 
select a.*, date(lmt.startApplyDateTime) application_date from 
(
(WITH parsed as (
select safe_cast(customerId as string) customerId, digitalLoanAccountId,modelDisplayName,modelVersionId,
safe_cast(prediction as numeric) prediction,trenchCategory,
REPLACE(REPLACE(calcFeature, "'", '"'), "None", "null") AS calcFeatures,
    deviceOs osType,
FROM `prj-prod-dataplatform.audit_balance.ml_model_run_details`
where modelDisplayName in ('Alpha-Cash-Stack-Model', 'alpha_stack_model_cash')
)
SELECT
    t.* EXCEPT(calcFeatures),
   
    SAFE_CAST(JSON_VALUE(calcFeatures, '$.apps_score') AS FLOAT64) 
        AS apps_score,

    SAFE_CAST(JSON_VALUE(calcFeatures, '$.c_demo_score') AS FLOAT64) 
        AS c_demo_score,

    SAFE_CAST(JSON_VALUE(calcFeatures, '$.c_credo_score') AS FLOAT64) 
        AS c_credo_score,

    SAFE_CAST(JSON_VALUE(calcFeatures, '$.c_tx_score') AS FLOAT64) 
        AS c_tx_score,

    SAFE_CAST(JSON_VALUE(calcFeatures, '$.ca_cic_score') AS FLOAT64) 
        AS ca_cic_score,

    SAFE_CAST(JSON_VALUE(calcFeatures, '$.c_event_score') AS FLOAT64) 
        AS c_event_score,

    SAFE_CAST(JSON_VALUE(calcFeatures, '$.c_device_score') AS FLOAT64) 
        AS c_device_score,
    'Prod' DataSelection
FROM  parsed t
where modelVersionId = 'v1.2'
)
union all 
(WITH parsed as (
select safe_cast(customerId as string) customerId, digitalLoanAccountId,modelDisplayName,modelVersionId,
safe_cast(prediction as numeric) prediction,trenchCategory,
REPLACE(REPLACE(calcFeature, "'", '"'), "None", "null") AS calcFeatures, Data_selection,
    deviceOs osType, 
FROM `prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116`
where modelDisplayName in ('Alpha-Cash-Stack-Model', 'alpha_stack_model_cash')
)
SELECT
    t.* EXCEPT(calcFeatures, Data_selection),


    SAFE_CAST(JSON_VALUE(calcFeatures, '$.apps_score') AS FLOAT64) 
        AS apps_score,

    SAFE_CAST(JSON_VALUE(calcFeatures, '$.c_demo_score') AS FLOAT64) 
        AS c_demo_score,

    SAFE_CAST(JSON_VALUE(calcFeatures, '$.c_credo_score') AS FLOAT64) 
        AS c_credo_score,

    SAFE_CAST(JSON_VALUE(calcFeatures, '$.c_tx_score') AS FLOAT64) 
        AS c_tx_score,

    SAFE_CAST(JSON_VALUE(calcFeatures, '$.ca_cic_score') AS FLOAT64) 
        AS ca_cic_score,

    SAFE_CAST(JSON_VALUE(calcFeatures, '$.c_event_score') AS FLOAT64) 
        AS c_event_score,

    SAFE_CAST(JSON_VALUE(calcFeatures, '$.c_device_score') AS FLOAT64) 
        AS c_device_score,
    Data_selection DataSelection
FROM  parsed t
where modelVersionId = 'v1.2'
)) a
left join `risk_credit_mis.loan_master_table` lmt on lmt.digitalLoanAccountId = a.digitalLoanAccountId
;
"""

query_job = client.query(sq)
# Wait until query finishes
query_job.result()
print("BigQuery job completed successfully!")


# ---------------------------------------------------------
# 3. Job details
# ---------------------------------------------------------
print("\n========== JOB DETAILS ==========")

print(f"Job ID              : {query_job.job_id}")
print(f"State               : {query_job.state}")
print(f"Created             : {query_job.created}")
print(f"Started             : {query_job.started}")
print(f"Ended               : {query_job.ended}")

# Bytes processed
if query_job.total_bytes_processed is not None:
    print(
        f"Bytes processed     : "
        f"{query_job.total_bytes_processed:,} "
        f"({query_job.total_bytes_processed / 1024**3:.3f} GB)"
    )

if query_job.total_bytes_billed is not None:
    print(
        f"Bytes billed        : "
        f"{query_job.total_bytes_billed:,} "
        f"({query_job.total_bytes_billed / 1024**3:.3f} GB)"
    )


# ---------------------------------------------------------
# 4. Get newly created table
# ---------------------------------------------------------
table_id = (
    "prj-prod-dataplatform."
    "dap_ds_poweruser_playground."
    "alpha_cash_stack_model_v1_2"
)

table = client.get_table(table_id)


# ---------------------------------------------------------
# 5. Table shape and metadata
# ---------------------------------------------------------
rows = table.num_rows
columns = len(table.schema)

print("\n========== TABLE DETAILS ==========")

print(f"Table               : {table_id}")
print(f"Shape               : ({rows:,}, {columns})")
print(f"Rows                : {rows:,}")
print(f"Columns             : {columns:,}")

print(
    f"Table size          : "
    f"{table.num_bytes:,} bytes "
    f"({table.num_bytes / 1024**2:.2f} MB)"
)

print(f"Created             : {table.created}")
print(f"Last modified       : {table.modified}")


# ---------------------------------------------------------
# 6. Print schema / column details
# ---------------------------------------------------------
print("\n========== SCHEMA ==========")

for i, field in enumerate(table.schema, start=1):
    print(
        f"{i:02d}. "
        f"{field.name:<30} "
        f"{field.field_type:<15} "
        f"{field.mode}"
    )

Starting BigQuery job...
BigQuery job completed successfully!

========== JOB DETAILS ==========
Job ID              : a407a512-8b44-4bd2-bd97-932f28e9daaa
State               : DONE
Created             : 2026-10-06 03:26:14.670000+00:00
Started             : 2026-10-06 03:26:14.923000+00:00
Ended               : 2026-10-06 03:26:18.043000+00:00
Bytes processed     : 10,770,806,976 (10.031 GB)
Bytes billed        : 10,770,972,672 (10.031 GB)

========== TABLE DETAILS ==========
Table               : prj-prod-dataplatform.dap_ds_poweruser_playground.alpha_cash_stack_model_v1_2
Shape               : (104,717, 16)
Rows                : 104,717
Columns             : 16
Table size          : 14,348,499 bytes (13.68 MB)
Created             : 2026-10-06 03:26:17.912000+00:00
Last modified       : 2026-10-06 03:26:17.913000+00:00

========== SCHEMA ==========
01. customerId                     STRING          NULLABLE
02. digitalLoanAccountId           STRING          NULLABLE
03. modelDisplay